#### 0. Generate Scene with MjSpec

In [12]:
import os
import sys
import numpy as np
import time

import mujoco

sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.SPEC_HELPER import *

In [13]:
spec_helper = MjSpecHelper()
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, 0.5, 0),
    r=(0, 0, -1.57),
    prefix="",
    suffix="_right"
)
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, -0.5, 0),
    r=(0, 0, 1.57),
    prefix="",
    suffix="_left"
)
spec_helper.add_geom(
    name="box",
    type='box',
    size=(0.15, 0.15, 0.15),
    freejoint = False,
    p=(0, 0, 0.2),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 0.3, 0.5),
    group=1,
    friction=(1.0, 0.005, 0.0001),
    mass=0.5
)
spec_helper.add_site(
    name="contact_left",
    # size=(0.01,0.01,0.01),
    size=(0.03,0.03,0.03),
    p=(0, 0.15, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="contact_right",
    size=(0.03,0.03,0.03),
    p=(0, -0.15, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)

model, data = spec_helper.compile()
spec_helper.save_to_xml("../asset/xml/scene_panda_lr.xml")

#### 1. Initialize Scene

In [14]:
body_names = get_body_names(model, data)
geom_names = get_geom_names(model, data)

In [15]:
joint_names = get_joint_names(model, data)
joint_names_left = [name for name in joint_names if name is not None and "_left" in name]
joint_names_right = [name for name in joint_names if name is not None and "_right" in name]

print("joint_names_left: ", joint_names_left)
print("joint_names_right: ", joint_names_right)

joint_names_left:  ['joint1_left', 'joint2_left', 'joint3_left', 'joint4_left', 'joint5_left', 'joint6_left', 'joint7_left']
joint_names_right:  ['joint1_right', 'joint2_right', 'joint3_right', 'joint4_right', 'joint5_right', 'joint6_right', 'joint7_right']


In [16]:
""" GO TO INITIAL QPOS """
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

In [17]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
viewer.view_site(group=0, show=True)
viewer.view_site(group=1, show=True)
viewer.view_site(group=2, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    mujoco.mj_step(model, data)
    # mujoco.mj_kinematics(model, data)
    # mujoco.mj_forward(model, data)
    viewer.render()

viewer.close()
del(viewer)